# FINAL 01 — Data Preparation and Frozen Linguistic Groups

**Thesis:** An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns  
**Student:** Abhishek Tomar | PN1196973 | LJMU | October 2026

**Purpose:** Rebuild final experimental data from the approved source partitions while applying the frozen EDA-01 linguistic rules consistently.

> **Final-run rule:** TweetEval test data is held out from model/feature/hyperparameter selection. External datasets are never used for source-model selection. Linguistic subgroup thresholds are frozen and are not tuned after observing subgroup sizes.

In [1]:
!pip install -q emoji vaderSentiment pandas pyarrow scikit-learn scipy google-api-python-client

from google.colab import drive, auth
drive.mount('/content/drive')

import io
import sys
from pathlib import Path
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Shared final-run utility stored in the Drive Experiments folder.
UTIL_FILE_ID = "14KfwtUf2zSd_jlW7i-6K8XgBJ62Aqk02"
UTIL_LOCAL = Path("/content/final_thesis_utils.py")

def load_final_thesis_utils():
    if not UTIL_LOCAL.exists() or UTIL_LOCAL.stat().st_size == 0:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        service = build("drive", "v3", credentials=creds, cache_discovery=False)
        request = service.files().get_media(fileId=UTIL_FILE_ID)
        with io.FileIO(str(UTIL_LOCAL), "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Downloading final_thesis_utils.py: {int(status.progress()*100)}%")

    assert UTIL_LOCAL.exists() and UTIL_LOCAL.stat().st_size > 0, \
        "final_thesis_utils.py could not be downloaded."

    if "/content" not in sys.path:
        sys.path.insert(0, "/content")

    import importlib
    import final_thesis_utils as ftu
    ftu = importlib.reload(ftu)

    RUN_ROOT = Path("/content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN")
    ftu.FINAL_ROOT = RUN_ROOT
    ftu.FINAL = {
        "root": RUN_ROOT,
        "data": RUN_ROOT / "data",
        "models": RUN_ROOT / "models",
        "predictions": RUN_ROOT / "predictions",
        "results": RUN_ROOT / "results",
        "figures": RUN_ROOT / "figures",
        "manifests": RUN_ROOT / "manifests",
    }
    return ftu

ftu = load_final_thesis_utils()

globals().update({k: v for k, v in vars(ftu).items() if not k.startswith("_")})
FINAL_ROOT = ftu.FINAL_ROOT
FINAL = ftu.FINAL
setup_final_dirs()

print("✅ final_thesis_utils imported successfully")
print("✅ Final run root:", FINAL_ROOT)

import json
import pandas as pd, numpy as np


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.2 MB/s eta 0:00:00
Mounted at /content/drive


✅ final_thesis_utils imported successfully
✅ Final run root: /content/drive/MyDrive/LJMU_FINAL_EXPERIMENT_RUN


## 1. Load approved source partitions
The existing source parquet files are treated as data inputs only. Any legacy subgroup labels are discarded and regenerated from raw `text` using the frozen rules.

In [2]:
tw_train=pd.read_parquet(LEGACY["data"]/"tw_train.parquet").copy()
tw_val=pd.read_parquet(LEGACY["data"]/"tw_val.parquet").copy()
tw_test=pd.read_parquet(LEGACY["data"]/"tw_test.parquet").copy()

for name,df,expected in [
    ("train",tw_train,45615),("validation",tw_val,2000),("test",tw_test,12284)
]:
    assert len(df)==expected, (name,len(df),expected)
    assert "text" in df.columns and "sentiment" in df.columns
    print(name,len(df))

train 45615
validation 2000
test 12284


In [3]:
def rebuild(df, split):
    x=df.copy()
    # Keep only required source columns plus any provenance columns.
    x["split"]=split
    x["sentiment"]=x["sentiment"].astype(str).str.lower()
    assert set(x["sentiment"].unique()).issubset(set(LABELS))
    x["label_id"]=x["sentiment"].map(LABEL_TO_ID).astype(int)
    x=tag_subgroups(x,"text")
    x["text_clean"]=x["text"].map(clean_text)
    return x

tw_train=rebuild(tw_train,"train")
tw_val=rebuild(tw_val,"validation")
tw_test=rebuild(tw_test,"test")

for name,df in [("train",tw_train),("validation",tw_val),("test",tw_test)]:
    print("\n",name)
    print(df["subgroup"].value_counts())


 train
subgroup
reference-unmarked    44938
other                   378
slang-heavy             204
sarcasm-indicated        95
Name: count, dtype: int64

 validation
subgroup
reference-unmarked    1971
other                   20
slang-heavy              6
sarcasm-indicated        3
Name: count, dtype: int64

 test
subgroup
reference-unmarked    10388
other                  1112
emoji-heavy             710
slang-heavy              60
sarcasm-indicated        14
Name: count, dtype: int64


## 2. Leakage and integrity controls
Exact-text overlap across official splits is measured and reported. It is **not silently removed** in this notebook; the audit is saved so the thesis can document the observed condition and any agreed handling.

In [4]:
def norm(t): return " ".join(str(t).strip().lower().split())
sets={k:set(v.text.map(norm)) for k,v in {"train":tw_train,"validation":tw_val,"test":tw_test}.items()}
overlaps=[
    {"pair":"train-validation","n_exact_text_overlap":len(sets["train"]&sets["validation"])},
    {"pair":"train-test","n_exact_text_overlap":len(sets["train"]&sets["test"])},
    {"pair":"validation-test","n_exact_text_overlap":len(sets["validation"]&sets["test"])},
]
overlap_df=pd.DataFrame(overlaps)
print(overlap_df)
overlap_df.to_csv(FINAL["results"]/"F01_split_overlap_audit.csv",index=False)

               pair  n_exact_text_overlap
0  train-validation                     0
1        train-test                     0
2   validation-test                     0


In [5]:
for name,df in [("train",tw_train),("validation",tw_val),("test",tw_test)]:
    df.to_parquet(FINAL["data"]/f"tweeteval_{name}_final.parquet",index=False)

test_counts=tw_test["subgroup"].value_counts().rename_axis("Subgroup").reset_index(name="N")
test_counts["Interpretation Status"]=np.where(test_counts["N"]>=MIN_STABLE_N,"stable_quantitative","exploratory_descriptive_only")
test_counts.to_csv(FINAL["results"]/"F01_test_subgroup_sufficiency.csv",index=False)
print(test_counts)

             Subgroup      N         Interpretation Status
0  reference-unmarked  10388           stable_quantitative
1               other   1112           stable_quantitative
2         emoji-heavy    710           stable_quantitative
3         slang-heavy     60           stable_quantitative
4   sarcasm-indicated     14  exploratory_descriptive_only


## Frozen interpretation
The `sarcasm-indicated` group is a surface-cue proxy, not a gold sarcasm label. Groups below `N=30` are reported descriptively/exploratorily and are never used to tune the subgroup rules.